In [ ]:
import sys, os, warnings
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import Markdown, display
import config
from src.utils import set_seed
set_seed(config.SEED)
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.facecolor": "#F5F1E8", "figure.facecolor": "#F5F1E8", "axes.edgecolor": "#77766D",
                     "font.size": 10, "axes.titleweight": "bold"})
PALETTE = ["#344C3D", "#BD755A", "#C4A46A", "#899982", "#77766D", "#242723"]
sns.set_palette(PALETTE)
def note(txt):  # data-driven observation rendered as markdown
    display(Markdown("> **Observation.** " + txt))

# 05 — Deep Learning Experiments: BiLSTM

A **bidirectional LSTM** sentence classifier in PyTorch, trained from scratch. It tests whether modelling *word order* (negation scope, "no significant difference") beats bag-of-n-grams TF-IDF.

Pipeline: regex tokenisation (`src.preprocessing.tokenize`, shared with the app) → vocabulary (top 30k words, `min_freq=2`, `<pad>`/`<unk>`) → integer padding → `Embedding(100)` → `BiLSTM(128 per direction)` → masked max-pool over time → dropout → linear(5).

**Compute compromise (CPU):** ablations (embedding init, sequence length, dropout) use a **30,000-sentence** subset and 3 epochs; the final model is trained on a **60,000-sentence** subset (up to 5 epochs, early stopping on dev macro-F1) rather than all 180k. The classical Logistic Regression on the same 30k subset (`tfidf_lr_30k`, notebook 04) is the like-for-like reference.

In [ ]:
import time, json, torch, torch.nn as nn
import src.data as D
from src.classification import build_vocab, encode_texts, make_bilstm, BiLSTMClassifier
from src.evaluation import classification_metrics, measure_latency_ms, softmax
from src.utils import save_json, update_json, load_json, timer
torch.set_num_threads(4); LAB = config.CLASS_LABELS; L2I = {l: i for i, l in enumerate(LAB)}
train, dev, test = (D.load_pubmed_rct(s) for s in ("train", "dev", "test"))
eval_sub = D.eval_subset(test)
sub30m = D.stratified_subset(train, 30_000, config.SEED); sub30 = sub30m
vocab = build_vocab(train.text, max_size=30_000, min_freq=2)
print("vocab size:", len(vocab), "| device:", "cpu (torch", torch.__version__ + ")")

## 1. Data pipeline: tokenisation, padding, embeddings

Sequences are truncated/padded to `max_len` (64 by default; EDA showed this covers ~99% of sentences). Padding index 0 is masked out of the max-pooling so padded positions never influence the sentence vector.

In [ ]:
ex = train.text.iloc[1]
print(ex); print(encode_texts([ex], vocab, 16)[0])
oov = np.mean([w not in vocab for t in dev.text.sample(5000, random_state=0) for w in __import__("src.preprocessing", fromlist=["tokenize"]).tokenize(t)])
print(f"dev OOV token rate with this vocabulary: {oov:.2%}")

## 2. Training loop with early stopping

In [ ]:
def batches(X, y, bs, shuffle):
    idx = np.random.permutation(len(X)) if shuffle else np.arange(len(X))
    for i in range(0, len(X), bs):
        j = idx[i:i + bs]; xb = X[j]; xb = xb[:, : max(1, int((xb != 0).sum(1).max()))]   # trim padding for speed
        yield torch.as_tensor(xb), torch.as_tensor(y[j])

def evaluate(model, X, y, bs=512):
    model.eval(); logits = []; loss = 0
    with torch.no_grad():
        for xb, yb in batches(X, y, bs, False):
            o = model(xb); loss += nn.functional.cross_entropy(o, yb, reduction="sum").item(); logits.append(o.numpy())
    lg = np.vstack(logits); return loss / len(X), lg

def run(name, data, max_len=64, epochs=3, pretrained=None, dropout=0.3, lr=2e-3, patience=2, bs=128, verbose=True):
    set_seed(config.SEED)
    Xtr, ytr = encode_texts(data.text.tolist(), vocab, max_len), data.label.map(L2I).values
    Xdv, ydv = encode_texts(dev.text.tolist(), vocab, max_len), dev.label.map(L2I).values
    model = make_bilstm(len(vocab), dropout=dropout, pretrained=pretrained)
    opt = torch.optim.Adam(model.parameters(), lr=lr); hist, best, best_state, bad = [], -1, None, 0
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train(); tl = tc = n = 0
        for xb, yb in batches(Xtr, ytr, bs, True):
            opt.zero_grad(); o = model(xb); loss = nn.functional.cross_entropy(o, yb); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tl += loss.item() * len(yb); tc += (o.argmax(1) == yb).sum().item(); n += len(yb)
        dl, lg = evaluate(model, Xdv, ydv); pred = lg.argmax(1)
        f1 = classification_metrics([LAB[i] for i in ydv], [LAB[i] for i in pred], LAB)["f1_macro"]
        hist.append({"epoch": ep, "train_loss": tl / n, "train_acc": tc / n, "dev_loss": dl, "dev_acc": float((pred == ydv).mean()), "dev_macro_f1": f1})
        if verbose: print(f"[{name}] epoch {ep} | train loss {tl/n:.4f} acc {tc/n:.4f} | dev loss {dl:.4f} acc {hist[-1]['dev_acc']:.4f} macro-F1 {f1:.4f} | {time.time()-t0:.0f}s")
        if f1 > best: best, bad, best_state = f1, 0, {k: v.clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: print("early stopping"); break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(hist), time.time() - t0

## 3. Ablations on a 30k subset (3 epochs each)

* **A** random embeddings, `max_len=64`, dropout 0.3 (baseline)
* **B** embeddings initialised from the **Word2Vec** vectors trained in notebook 03 (TF-IDF-vs-embedding question, now inside a sequence model)
* **C** `max_len=32` (sequence-length experiment)
* **D** dropout 0.0 (overfitting / regularisation experiment)

In [ ]:
from gensim.models import Word2Vec
w2v_path = config.DATA_PROCESSED / "word2vec.model"
if w2v_path.exists(): w2v = Word2Vec.load(str(w2v_path)); print("loaded Word2Vec from notebook 03")
else:
    from src.features import train_word2vec; w2v = train_word2vec(train.text, epochs=5); print("trained Word2Vec here")
emb = np.random.default_rng(0).normal(0, 0.1, (len(vocab), 100)).astype("float32"); emb[0] = 0; hit = 0
for w, i in vocab.items():
    if w in w2v.wv: emb[i] = w2v.wv[w]; hit += 1
print(f"Word2Vec covers {hit / len(vocab):.1%} of the BiLSTM vocabulary")
ABL = {}
for key, kw in {"A random-init": {}, "B word2vec-init": {"pretrained": emb}, "C max_len=32": {"max_len": 32}, "D dropout=0": {"dropout": 0.0}}.items():
    m, h, secs = run(key, sub30, epochs=3, verbose=False, **kw); ABL[key] = {"best_dev_macro_f1": float(h.dev_macro_f1.max()), "final_train_acc": float(h.train_acc.iloc[-1]), "final_dev_acc": float(h.dev_acc.iloc[-1]), "seconds": secs}
abl = pd.DataFrame(ABL).T; display(abl.round(4)); update_json(config.ART_CLS / "ablations.json", "bilstm", abl.round(4).to_dict("index"))
w = abl.best_dev_macro_f1
note(f"Word2Vec initialisation changes dev macro-F1 by {w['B word2vec-init'] - w['A random-init']:+.4f} versus random init; halving the sequence length to 32 changes it by {w['C max_len=32'] - w['A random-init']:+.4f}; "
     f"removing dropout changes it by {w['D dropout=0'] - w['A random-init']:+.4f}. Train-minus-dev accuracy gap: " + ", ".join(f"{k}: {abl.loc[k,'final_train_acc'] - abl.loc[k,'final_dev_acc']:+.3f}" for k in abl.index) + " (train accuracy is measured with dropout active).")

## 4. Final model

Trained on the 30k subset with the best embedding initialisation from the ablation (ties → random init), up to 5 epochs, early stopping (patience 2) on dev macro-F1.

In [ ]:
use_w2v = w["B word2vec-init"] > w["A random-init"] + 0.002
model, hist, train_secs = run("final", sub30m, epochs=5, pretrained=emb if use_w2v else None)
print("initialisation:", "word2vec" if use_w2v else "random", f"| training {train_secs:.0f}s")
hist

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
axes[0].plot(hist.epoch, hist.train_loss, "o-", label="train", color=PALETTE[0]); axes[0].plot(hist.epoch, hist.dev_loss, "o-", label="dev", color=PALETTE[1]); axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(hist.epoch, hist.train_acc, "o-", label="train (dropout on)", color=PALETTE[0]); axes[1].plot(hist.epoch, hist.dev_acc, "o-", label="dev", color=PALETTE[1]); axes[1].set_title("Accuracy"); axes[1].legend()
axes[2].plot(hist.epoch, hist.dev_macro_f1, "o-", color=PALETTE[2]); axes[2].set_title("Dev macro-F1")
for a in axes: a.set_xlabel("epoch"); a.set_xticks(hist.epoch)
plt.tight_layout(); plt.show()
bi = int(hist.dev_macro_f1.idxmax()); gap = hist.train_acc.iloc[bi] - hist.dev_acc.iloc[bi]
note(f"Best epoch by dev macro-F1: {int(hist.epoch.iloc[bi])} (dev macro-F1 {hist.dev_macro_f1.max():.4f}). " +
     ("Dev loss rises while train loss keeps falling in later epochs — the classic overfitting signature, which early stopping guards against." if (hist.dev_loss.iloc[-1] > hist.dev_loss.min() + 1e-3 and len(hist) > 1) else "Dev loss did not diverge from train loss within the epochs run, so no strong overfitting is visible at this budget (more epochs might reveal it).") +
     f" Train–dev accuracy gap at the best epoch: {gap:+.3f}.")

## 5. Test evaluation, artifact export and comparison

In [ ]:
d = config.ART_CLS / "bilstm"; d.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), d / "model.pt"); save_json(vocab, d / "vocab.json"); save_json({"max_len": 64, "emb_dim": 100, "hidden": 128, "init": "word2vec" if use_w2v else "random"}, d / "config.json")
clf = BiLSTMClassifier("bilstm", d)          # round-trip: reload exactly what the Flask app will load
P_full = clf.predict_proba(test.text.tolist()); P_sub = clf.predict_proba(eval_sub.text.tolist())
pf = [LAB[i] for i in P_full.argmax(1)]; ps = [LAB[i] for i in P_sub.argmax(1)]
res = {"kind": "deep", "label": "BiLSTM", "train_n": len(sub30m), "train_seconds": train_secs, "epochs_run": len(hist), "params": sum(p.numel() for p in model.parameters()),
       "dev": classification_metrics(dev.label, [LAB[i] for i in clf.predict_proba(dev.text.tolist()).argmax(1)], LAB), "test_full": classification_metrics(test.label, pf, LAB)}
res.update(classification_metrics(eval_sub.label, ps, LAB))
res["latency_ms"] = measure_latency_ms(lambda b: clf.predict_proba(b), eval_sub.text.tolist()[:1000], repeats=3)
res["model_mb"] = (d / "model.pt").stat().st_size / 1e6
np.savez_compressed(config.ART_EVAL / "preds_bilstm.npz", probs=P_sub.astype("float32"), y=eval_sub.label.map(L2I).values)
update_json(config.ART_CLS / "metrics.json", "bilstm", res); save_json(hist.to_dict("records"), config.ART_EVAL / "bilstm_history.json")
M = load_json(config.ART_CLS / "metrics.json")
cmp_ = pd.DataFrame({k: {"model": M[k]["label"], "train n": M[k]["train_n"], "test macro-F1": M[k]["test_full"]["f1_macro"], "test accuracy": M[k]["test_full"]["accuracy"], "train s": M[k]["train_seconds"], "ms/sent": M[k]["latency_ms"], "MB": M[k]["model_mb"]}
                     for k in ["tfidf_lr", "tfidf_lr_30k", "tfidf_svm", "tfidf_nb", "bilstm"] if k in M}).T.set_index("model"); display(cmp_.round(4))
lr60 = M.get("tfidf_lr_30k", {}).get("test_full", {}).get("f1_macro", float("nan")); bi_f1 = res["test_full"]["f1_macro"]
note(f"BiLSTM test macro-F1 = {bi_f1:.4f} (accuracy {res['test_full']['accuracy']:.4f}) vs Logistic Regression on the same 30k sentences = {lr60:.4f} (Δ {bi_f1 - lr60:+.4f}). "
     f"Cost: {res['params']:,} parameters, {res['latency_ms']:.2f} ms/sentence, {train_secs:.0f}s training. Whether sequence modelling pays off here is answered by the sign and size of Δ; "
     f"the bidirectional encoder can read negation scope but, like TF-IDF, still sees each sentence without its abstract context.")

## Conclusions

* A BiLSTM is a valid, cheap deep baseline; early stopping on dev macro-F1 and a held-out test evaluation avoid optimistic reporting.
* Ablations (embedding initialisation, sequence length, dropout) are measured on a smaller budget and should be read as *directional*, not definitive.
* The model is exported to `artifacts/classification/bilstm/` and re-loaded through the same wrapper used by the Flask app, which verifies the inference path end to end.